## YouTube Lecture Transcription (GPU)

Transcribes audio files using Whisper on GPU.
Produces JSON transcripts compatible with the indexing pipeline.

**Prerequisites:**
1. Download audio locally: `uv run --group indexer python scripts/transcribe.py --download-only`
2. Zip the audio: `cd data && zip -r audio.zip audio/ && cd ..`
3. Upload `audio.zip` to Colab using the cell below

**Runtime:** Set to GPU via Runtime > Change runtime type > T4 GPU.

**Output:** JSON files in `transcripts/` -- download and copy to `data/transcripts/` in the repo.

### Setup

In [ ]:
!pip install -q openai-whisper

In [ ]:
import json
import zipfile
from pathlib import Path

import whisper

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Upload Audio

Upload audio files to the `audio/` directory. Two options:
- **Option A**: Run the cell below to upload `audio.zip` via code
- **Option B**: Drag-drop mp3 files (or `audio.zip`) into the `audio/` folder in the Colab file browser (left panel), then skip the upload cell

In [ ]:
AUDIO_DIR = Path("audio")
TRANSCRIPT_DIR = Path("transcripts")
AUDIO_DIR.mkdir(exist_ok=True)
TRANSCRIPT_DIR.mkdir(exist_ok=True)

In [ ]:
# Option A: upload audio.zip via code (skip this cell if using drag-drop)
from google.colab import files as colab_files  # type: ignore[import-not-found]

print("Upload audio.zip:")
uploaded = colab_files.upload()

for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name, "r") as zf:
            zf.extractall(".")
        print(f"Extracted {name}")

In [ ]:
# List audio files found (works for both upload and drag-drop)
audio_files = sorted(AUDIO_DIR.glob("*.mp3"))
print(f"Found {len(audio_files)} audio files:")
for f in audio_files:
    print(f"  {f.name}")

### Configuration

In [ ]:
WHISPER_MODEL = "large-v3"
LANGUAGE = "en"
CONDITION_ON_PREVIOUS_TEXT = False
NO_SPEECH_THRESHOLD = 0.6

# Domain terms to improve name/term spelling.
# Copy-paste content from data/whisper_prompt.md between the triple quotes.
INITIAL_PROMPT = """
Biplov Bhandari, Tim Mayer, David Saah, Lillianne Thomas, Chishan Zhang,
Wei Ji Leong, Daniel Weismann, Soumya Ranjan, Biplov, Muhammad Warizmi Wafiq,
Ate Poortinga, EarthRISE, NASA, Earth Resources Technology, University of San Francisco,
University of Alabama in Huntsville, Spatial Informatics Group,
U-Net, semantic segmentation, object detection, transfer learning,
Bayesian Neural Networks, foundation models, Clay Foundation Model,
remote sensing, Earth observation, deep learning, convolutional neural network,
Quarto, Qdrant, Bhutan, deforestation, phenology, soybean yield prediction,
active fire detection, downscaling, image classification, data fusion,
GeoTIFF, raster, MODIS, Landsat, Sentinel, Google Earth Engine
""".strip()

### Load Whisper Model

This downloads the model weights (~2.88 GB) on first run.

In [ ]:
model = whisper.load_model(WHISPER_MODEL)
print(f"Model loaded: {WHISPER_MODEL}")
print(f"Device: {next(model.parameters()).device}")

### Filter Audio Files (Optional)

Skip files that already have transcripts, or select specific video IDs.

In [ ]:
FORCE = True
ONLY_IDS = []  # e.g. ["l3Tv5UBlw9I", "DnPosJ6IJ0Y"]
SKIP_IDS = ["INDi5PrcRTY"]

to_process = []
for audio_file in audio_files:
    video_id = audio_file.stem
    if video_id in SKIP_IDS:
        print(f"  Skipping {video_id} (in SKIP_IDS)")
        continue
    if ONLY_IDS and video_id not in ONLY_IDS:
        continue
    transcript_path = TRANSCRIPT_DIR / f"{video_id}.json"
    if transcript_path.exists() and not FORCE:
        print(f"  Skipping {video_id}: transcript exists")
        continue
    to_process.append(audio_file)

print(f"Will transcribe {len(to_process)} files")

### Transcribe

Runs Whisper on each audio file and saves JSON transcripts.

In [ ]:
YOUTUBE_WATCH_URL = "https://www.youtube.com/watch?v="


def _load_video_meta(video_id):
    """Load metadata saved by transcribe.py --download-only, if available."""
    meta_path = AUDIO_DIR / f"{video_id}.meta.json"
    if meta_path.exists():
        with open(meta_path) as f:
            return json.load(f)
    return None


def transcribe_and_save(audio_path):
    video_id = audio_path.stem
    result = model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        initial_prompt=INITIAL_PROMPT,
        condition_on_previous_text=CONDITION_ON_PREVIOUS_TEXT,
        no_speech_threshold=NO_SPEECH_THRESHOLD,
    )
    raw_segments: list[dict] = result.get("segments", [])  # type: ignore[assignment]
    segments = [{"start": s["start"], "end": s["end"], "text": s["text"]} for s in raw_segments]

    # Use metadata from --download-only if available, otherwise fall back
    meta = _load_video_meta(video_id)
    duration = segments[-1]["end"] if segments else 0.0

    data = {
        "video_id": video_id,
        "title": meta["title"] if meta else video_id,
        "duration_seconds": meta["duration"] if meta else duration,
        "url": meta["url"] if meta else f"{YOUTUBE_WATCH_URL}{video_id}",
        "segments": segments,
    }
    output_path = TRANSCRIPT_DIR / f"{video_id}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    return output_path, len(segments)

In [ ]:
results = {"success": [], "failed": []}

for i, audio_file in enumerate(to_process, 1):
    video_id = audio_file.stem
    print(f"\n[{i}/{len(to_process)}] {video_id}")
    try:
        print("  Transcribing...")
        saved, n_segments = transcribe_and_save(audio_file)
        print(f"  Saved {n_segments} segments to {saved}")
        results["success"].append(video_id)
    except Exception as e:
        print(f"  FAILED: {e}")
        results["failed"].append(video_id)

print("\n=== Done ===")
print(f"Success: {len(results['success'])}")
print(f"Failed: {len(results['failed'])}")
if results["failed"]:
    print(f"Failed IDs: {results['failed']}")

### Post-Processing Corrections

Fix known Whisper misspellings across all transcripts.
Reads corrections from `data/transcript_corrections.yml` (upload it alongside `audio.zip`, or edit the dict below).

In [ ]:
import glob

# Try to load corrections from the YAML file; fall back to inline dict
CORRECTIONS_FILE = Path("transcript_corrections.yml")
if not CORRECTIONS_FILE.exists():
    CORRECTIONS_FILE = AUDIO_DIR / "transcript_corrections.yml"

if CORRECTIONS_FILE.exists():
    import yaml  # type: ignore[import-not-found]
    with open(CORRECTIONS_FILE) as f:
        corrections = yaml.safe_load(f) or {}
    print(f"Loaded {len(corrections)} corrections from {CORRECTIONS_FILE}")
else:
    corrections = {
        "Dip Loth": "Biplov",
        "Vip Loth": "Biplov",
        "Bipalap": "Biplov",
        "Bipalav": "Biplov",
        "Biplav": "Biplov",
        "bib love": "Biplov",
        "biblov": "Biplov",
        "Piplov": "Biplov",
    }
    print(f"No corrections file found, using {len(corrections)} inline corrections")

for f in sorted(glob.glob(str(TRANSCRIPT_DIR / "*.json"))):
    text = open(f).read()
    changed = False
    for wrong, right in corrections.items():
        if wrong in text:
            text = text.replace(wrong, str(right))
            changed = True
    if changed:
        open(f, "w").write(text)
        print(f"Fixed: {Path(f).name}")

print("Done.")

### Verify a Transcript

Spot-check a transcript to verify quality.

In [ ]:
transcripts = sorted(TRANSCRIPT_DIR.glob("*.json"))
if not transcripts:
    print("No transcripts found.")
else:
    sample = transcripts[0]
    with open(sample) as f:
        data = json.load(f)
    print(f"Video: {data['video_id']}")
    print(f"Duration: {data['duration_seconds']:.0f}s")
    print(f"Segments: {len(data['segments'])}")
    print("\nFirst 10 segments:")
    for seg in data["segments"][:10]:
        print(f"  [{seg['start']:.1f}-{seg['end']:.1f}] {seg['text']}")

### Download Transcripts

Download the `transcripts/` folder and copy the JSON files to `data/transcripts/` in the repo.

In [ ]:
import shutil
from google.colab import files  # type: ignore[import-not-found]  # noqa: E402

shutil.make_archive("transcripts", "zip", ".", "transcripts")
files.download("transcripts.zip")